# 欢迎进入 VirtAI Notebook
您可以在这里编辑代码和文档，在编辑代码前您需阅读本文了解文件目录的作用及权限，以免使用不当造成数据丢失。

## 关于文件目录



| 存储 | 路径 | 环境变量($+变量) | 权限 | 大小 | 备注 |
|-------|-------|-------|-------|-------|-------|
| 容器储存 | 代码、数据集、模型、结果集<br>路径以外的所有路径 | - |【开发环境】可读写<br>【离线训练】可读写<br>【推理服务】可读写 | small：20G<br>midium:30G<br>large：50G<br>Xlarge:100G | 1.[临时保存]把容器保存成新<br>镜像 (包含容器存储的数据)<br>2.容器关闭或重启，会销毁<br>容器，容器存储的数据不会保留。 |
| 代码 | /gemini/code | GEMINI_CODE | 【开发环境】可读写<br>【离线训练】只读<br>【推理服务】只读 | 不限制大小 | 1.在项目内挂载，归属于所在的项目<br>2.启动容器后，如果开启了SSH或<br>注入了JupyterLab，可以通过SSH工具<br>或JupyterLab上传下载 |
| 数据集 | /gemini/data-1<br>/gemini/data-2<br>/gemini/data-3 | GEMINI_DATA_IN1<br>GEMINI_DATA_IN2<br>GEMINI_DATA_IN3 | 只读 | 不限制大小 | 在【数据】栏内上传数据，保存在数据<br>目录下，创建项目时选择会挂载到容器内。 |
| 模型 | /gemini/pretrain<br>/gemini/pretrain2<br>/gemini/pretrain3 | GEMINI_PRETRAIN<br>GEMINI_PRETRAIN2<br>GEMINI_PRETRAIN3 | 只读 | 不限制大小 | - |
| 结果集 | /gemini/output | GEMINI_DATA_OUT | 仅【离线训练】有此功能<br>可读写 | 不限制大小 | 挂载在项目内，归属于所在的项目 |

In [ ]:
# 试试这个经典示例
print ("Hello VirtAI")

In [ ]:
# 查看项目代码
!ls /gemini/code
!ls $GEMINI_CODE

In [ ]:
# 查看挂载的数据集
!ls /gemini/data-1
!ls $GEMINI_DATA_IN1

In [ ]:
# 查看挂载的预训练数据/模型
!ls /gemini/pretrain
!ls $GEMINI_PRETRAIN

In [1]:
pip install -r requirements.txt

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 23.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
!python cervical_cancer_optimized.py

使用设备: cuda
正在加载数据集...
总共加载 1834 张图像

各类别样本分布:
  浅表鳞状上皮: 148
  中度鳞状上皮: 140
  柱状上皮: 196
  轻度发育不良: 364
  中度发育不良: 292
  重度发育不良: 394
  原位癌: 300

数据划分:
  训练集: 1375
  验证集: 229
  测试集: 230

模型结构:
CervicalCNN(
  (stem): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): ConvBlock(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
      )
      (conv2): ConvBlock(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, moment

In [3]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 数据路径配置
DATA_DIR = '实训任务3数据集'

# 类别映射
CLASS_NAMES = {
    'normal_superficiel': 0,
    'normal_intermediate': 1,
    'normal_columnar': 2,
    'light_dysplastic': 3,
    'moderate_dysplastic': 4,
    'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}

CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}

# 类别中文名称
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮',
    'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮',
    'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良',
    'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 根据数据分析得到的类别权重（基于统计结果）
CLASS_WEIGHTS = torch.tensor([2.66, 2.81, 2.01, 1.08, 1.35, 1.00, 1.31], dtype=torch.float32)


class FocalLoss(nn.Module):
    """
    Focal Loss implementation for handling class imbalance and hard examples
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            ce_loss = alpha_t * ce_loss
        
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


def load_data():
    """加载数据集"""
    print("正在加载数据集...")
    
    image_paths = []
    labels = []
    
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    
    print(f"总共加载 {len(image_paths)} 张图像")
    
    # 统计各类别样本数
    class_counts = {}
    for label in labels:
        class_name = CLASS_NAMES_REV[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    
    print("\n各类别样本分布:")
    for class_name, count in class_counts.items():
        print(f"  {CLASS_NAMES_CN[class_name]}: {count}")
    
    return image_paths, labels, class_counts


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
    
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = ConvBlock(in_channels, out_channels, stride=stride)
        self.conv2 = ConvBlock(out_channels, out_channels)
        
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        
        if self.downsample is not None:
            residual = self.downsample(x)
        
        out = out + residual
        return nn.ReLU(inplace=False)(out)


class CervicalCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        
        # 初始层
        self.stem = nn.Sequential(
            ConvBlock(3, 64, kernel_size=7, stride=2, padding=3),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        # Residual layers
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )
        
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
            ResidualBlock(128, 128),
            ResidualBlock(128, 128)
        )
        
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256),
            ResidualBlock(256, 256),
            ResidualBlock(256, 256),
            ResidualBlock(256, 256),
            ResidualBlock(256, 256)
        )
        
        self.layer4 = nn.Sequential(
            ResidualBlock(256, 512, stride=2),
            ResidualBlock(512, 512),
            ResidualBlock(512, 512)
        )
        
        # 全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=False),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=False),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, device='cuda'):
    model.to(device)
    best_val_f1 = 0.0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    print(f"\n开始训练，共 {num_epochs} 轮...")
    print("使用 Focal Loss + WeightedRandomSampler + 类别权重优化...")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_train / total_train
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)
        
        # 验证阶段
        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_running_loss += loss.item() * inputs.size(0)
                
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())
        
        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = correct_val / total_val
        
        # 计算验证集的F1分数
        precision, recall, f1, _ = precision_recall_fscore_support(all_val_labels, all_val_preds, average='weighted', zero_division=0)
        
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {f1:.4f}")
        
        # 保存最佳模型（基于F1分数）
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': f1
            }, 'best_cervical_model_optimized.pth')
            print(f"  → 保存最佳模型 (Epoch {best_epoch+1}, Val F1: {best_val_f1:.4f}, Acc: {best_val_acc:.4f})")
        
        # 更新学习率
        scheduler.step(f1)
    
    print(f"\n训练完成！")
    print(f"最佳 Epoch: {best_epoch+1}")
    print(f"最佳 Val F1: {best_val_f1:.4f}, Val Acc: {best_val_acc:.4f}")
    
    return model, train_losses, val_losses, train_accs, val_accs


def evaluate_model(model, test_loader, device='cuda'):
    checkpoint = torch.load('best_cervical_model_optimized.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    print("\n开始评估模型...")
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # 详细分类报告
    print("\n" + "=" * 100)
    print("【七分类详细评估指标】")
    print("=" * 100)
    report = classification_report(all_labels, all_preds, 
                               target_names=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                               digits=4,
                               zero_division=0)
    print(report)
    
    # 计算每类的详细指标
    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, zero_division=0)
    print("\n【各类别详细指标】")
    print(f"{'类别':<20} {'精确率(Precision)':<15} {'召回率(Recall)':<15} {'F1分数':<10} {'样本数':<10}")
    print("-" * 100)
    for i, class_name in enumerate(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES_CN[class_name]:<20} {precision[i]:<15.4f} {recall[i]:<15.4f} {f1[i]:<10.4f} {support[i]:<10}")
    
    # 二分类评估（正常/异常）
    binary_preds = [1 if p >= 3 else 0 for p in all_preds]
    binary_labels = [1 if l >= 3 else 0 for l in all_labels]
    
    print("\n" + "=" * 100)
    print("【二分类（正常/异常）评估指标】")
    print("=" * 100)
    binary_report = classification_report(binary_labels, binary_preds, 
                                          target_names=['正常', '异常'],
                                          digits=4,
                                          zero_division=0)
    print(binary_report)
    
    # 绘制混淆矩阵
    plt.figure(figsize=(14, 12))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()], 
                yticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                annot_kws={'size': 12})
    plt.title('混淆矩阵 - 七分类', fontsize=16, fontweight='bold')
    plt.xlabel('预测类别', fontsize=14)
    plt.ylabel('真实类别', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix_optimized.png', dpi=300, bbox_inches='tight')
    print("\n混淆矩阵已保存为 confusion_matrix_optimized.png")
    
    return all_preds, all_labels, all_probs


def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Loss曲线
    axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2, color='#1f77b4')
    axes[0, 0].plot(val_losses, label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[0, 0].set_title('训练/验证 Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].legend(fontsize=11)
    axes[0, 0].grid(True, alpha=0.3)
    
    # 准确率曲线
    axes[0, 1].plot(train_accs, label='Train Acc', linewidth=2, color='#1f77b4')
    axes[0, 1].plot(val_accs, label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[0, 1].set_title('训练/验证 准确率', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Accuracy', fontsize=12)
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 放大的Loss曲线（后半段）
    mid = len(train_losses) // 2
    axes[1, 0].plot(range(mid, len(train_losses)), train_losses[mid:], label='Train Loss', linewidth=2, color='#1f77b4')
    axes[1, 0].plot(range(mid, len(val_losses)), val_losses[mid:], label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[1, 0].set_title('训练/验证 Loss (后半段)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Loss', fontsize=12)
    axes[1, 0].legend(fontsize=11)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 放大的准确率曲线（后半段）
    axes[1, 1].plot(range(mid, len(train_accs)), train_accs[mid:], label='Train Acc', linewidth=2, color='#1f77b4')
    axes[1, 1].plot(range(mid, len(val_accs)), val_accs[mid:], label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[1, 1].set_title('训练/验证 准确率 (后半段)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Accuracy', fontsize=12)
    axes[1, 1].legend(fontsize=11)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history_optimized.png', dpi=300, bbox_inches='tight')
    print("训练历史曲线已保存为 training_history_optimized.png")


def main():
    # 确定设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 加载数据
    image_paths, labels, class_counts = load_data()
    
    # 数据划分 (训练集 75%, 验证集 12.5%, 测试集 12.5%)
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.25, random_state=42, stratify=labels)
    
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
    
    print(f"\n数据划分:")
    print(f"  训练集: {len(train_paths)}")
    print(f"  验证集: {len(val_paths)}")
    print(f"  测试集: {len(test_paths)}")
    
    # 更强的数据增强
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(45),
        #transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
        #transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
        #transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    
    # 创建数据集
    train_dataset = CervicalDataset(train_paths, train_labels, train_transform)
    val_dataset = CervicalDataset(val_paths, val_labels, val_test_transform)
    test_dataset = CervicalDataset(test_paths, test_labels, val_test_transform)
    
    # 使用 WeightedRandomSampler 处理类别不平衡
    train_labels_np = np.array(train_labels)
    class_sample_counts = np.array([np.sum(train_labels_np == i) for i in range(7)])
    weights = 1. / torch.tensor(class_sample_counts, dtype=torch.float)
    # 对样本较少的类别给予更高的采样权重
    severe_idx = CLASS_NAMES['severe_dysplastic']
    weights[severe_idx] *= 1.5  # 给重度发育不良更高的采样权重
    samples_weights = weights[train_labels_np]
    sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)
    
    # 创建自定义ResNet模型
    model = CervicalCNN(num_classes=7)
    print("\n模型结构:")
    print(model)
    
    # 使用 Focal Loss
    print("\n使用 Focal Loss (gamma=2.0, alpha=类别权重)")
    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(device), gamma=2.0, reduction='mean')
    
    # 优化器和学习率调度器
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    
    # 训练模型
    model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, device=device)
    
    # 绘制训练历史
    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    
    # 在测试集上评估
    evaluate_model(model, test_loader, device=device)
    
    print("\n" + "=" * 100)
    print("【最终优化版本】任务完成！使用 Focal Loss + WeightedRandomSampler + 类别权重优化")
    print("=" * 100)


if __name__ == "__main__":
    main()


使用设备: cuda
正在加载数据集...
总共加载 1834 张图像

各类别样本分布:
  浅表鳞状上皮: 148
  中度鳞状上皮: 140
  柱状上皮: 196
  轻度发育不良: 364
  中度发育不良: 292
  重度发育不良: 394
  原位癌: 300

数据划分:
  训练集: 1375
  验证集: 229
  测试集: 230

模型结构:
CervicalCNN(
  (stem): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): ConvBlock(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU()
      )
      (conv2): ConvBlock(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, moment

In [2]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 数据路径配置
DATA_DIR = '实训任务3数据集'

# 类别映射（不变）
CLASS_NAMES = {
    'normal_superficiel': 0,
    'normal_intermediate': 1,
    'normal_columnar': 2,
    'light_dysplastic': 3,
    'moderate_dysplastic': 4,
    'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}

CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}

CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮',
    'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮',
    'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良',
    'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# ========== 关键改动1：修正类别权重（强化困难类别） ==========
# 原权重严重低估了重度发育不良的损失贡献，现根据F1表现反推惩罚力度
CLASS_WEIGHTS = torch.tensor([1.5, 1.5, 2.5, 2.0, 3.5, 4.5, 2.0], dtype=torch.float32)

# ========== Focal Loss（保持不变，但alpha使用新权重） ==========
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            ce_loss = alpha_t * ce_loss
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== 数据集类（不变） ==========
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# ========== 加载数据函数（不变） ==========
def load_data():
    print("正在加载数据集...")
    image_paths = []
    labels = []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总共加载 {len(image_paths)} 张图像")
    class_counts = {}
    for label in labels:
        class_name = CLASS_NAMES_REV[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    print("\n各类别样本分布:")
    for class_name, count in class_counts.items():
        print(f"  {CLASS_NAMES_CN[class_name]}: {count}")
    return image_paths, labels, class_counts

# ========== 关键改动2：替换为预训练ResNet50 ==========
class CervicalCNN_Pretrained(nn.Module):
    def __init__(self, num_classes=7, pretrained_path='resnet50.pth'):
        super().__init__()
        # 加载ResNet50架构（不自动下载权重）
        self.backbone = models.resnet50(weights=None)
        # 加载本地预训练权重（strict=False 忽略分类头不匹配）
        if os.path.exists(pretrained_path):
            state_dict = torch.load(pretrained_path, map_location='cpu')
            # 如果权重是完整模型（含fc），需要移除fc层的权重
            if 'fc.weight' in state_dict:
                del state_dict['fc.weight']
                del state_dict['fc.bias']
            self.backbone.load_state_dict(state_dict, strict=False)
            print(f"成功加载本地预训练权重: {pretrained_path}")
        else:
            print(f"警告: 未找到 {pretrained_path}，将使用随机初始化（不推荐）")
        
        # 冻结前3个stage（保留底层特征），微调stage4和fc
        for name, param in self.backbone.named_parameters():
            if 'layer1' in name or 'layer2' in name or 'layer3' in name:
                param.requires_grad = False
        # 替换分类头
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# ========== CutMix 辅助函数 ==========
def cutmix_data(x, y, alpha=0.5):
    """对batch执行CutMix增强，返回混合后的图像、标签和lambda"""
    if not torch.is_tensor(x) or np.random.rand() > 0.5:
        return x, y, y, 1.0  # 不应用CutMix
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

# ========== 训练函数（修改：支持CutMix + OneCycleLR） ==========
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=60, device='cuda'):
    model.to(device)
    best_val_f1 = 0.0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    
    print(f"\n开始训练，共 {num_epochs} 轮...")
    print("使用 Focal Loss + 修正类别权重 + CutMix + OneCycleLR")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # ---------- CutMix ----------
            inputs, labels_a, labels_b, lam = cutmix_data(inputs, labels, alpha=0.5)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            # 混合损失
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # 更新学习率（OneCycleLR 在每个batch后step）
            scheduler.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_train / total_train
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)
        
        # 验证阶段（不变）
        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        all_val_preds, all_val_labels = [], []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())
        
        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = correct_val / total_val
        _, _, f1, _ = precision_recall_fscore_support(all_val_labels, all_val_preds, average='weighted', zero_division=0)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {f1:.4f}")
        
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': f1
            }, 'best_cervical_model_optimized.pth')
            print(f"  → 保存最佳模型 (Epoch {best_epoch+1}, Val F1: {best_val_f1:.4f}, Acc: {best_val_acc:.4f})")
    
    print(f"\n训练完成！最佳 Epoch: {best_epoch+1}, Val F1: {best_val_f1:.4f}, Val Acc: {best_val_acc:.4f}")
    return model, train_losses, val_losses, train_accs, val_accs

# ========== 评估函数（完全不变，保持原有输出） ==========
def evaluate_model(model, test_loader, device='cuda'):
    checkpoint = torch.load('best_cervical_model_optimized.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    print("\n开始评估模型...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    print("\n" + "=" * 100)
    print("【七分类详细评估指标】")
    print("=" * 100)
    report = classification_report(all_labels, all_preds, 
                               target_names=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                               digits=4, zero_division=0)
    print(report)
    
    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, zero_division=0)
    print("\n【各类别详细指标】")
    print(f"{'类别':<20} {'精确率(Precision)':<15} {'召回率(Recall)':<15} {'F1分数':<10} {'样本数':<10}")
    print("-" * 100)
    for i, class_name in enumerate(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES_CN[class_name]:<20} {precision[i]:<15.4f} {recall[i]:<15.4f} {f1[i]:<10.4f} {support[i]:<10}")
    
    binary_preds = [1 if p >= 3 else 0 for p in all_preds]
    binary_labels = [1 if l >= 3 else 0 for l in all_labels]
    print("\n" + "=" * 100)
    print("【二分类（正常/异常）评估指标】")
    print("=" * 100)
    binary_report = classification_report(binary_labels, binary_preds, target_names=['正常', '异常'], digits=4, zero_division=0)
    print(binary_report)
    
    plt.figure(figsize=(14, 12))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()], 
                yticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                annot_kws={'size': 12})
    plt.title('混淆矩阵 - 七分类 (优化后)', fontsize=16, fontweight='bold')
    plt.xlabel('预测类别', fontsize=14)
    plt.ylabel('真实类别', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix_optimized.png', dpi=300, bbox_inches='tight')
    print("\n混淆矩阵已保存为 confusion_matrix_optimized.png")
    return all_preds, all_labels, all_probs

# ========== 绘图函数（不变） ==========
def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2, color='#1f77b4')
    axes[0, 0].plot(val_losses, label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[0, 0].set_title('训练/验证 Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(train_accs, label='Train Acc', linewidth=2, color='#1f77b4')
    axes[0, 1].plot(val_accs, label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[0, 1].set_title('训练/验证 准确率', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
    mid = len(train_losses) // 2
    axes[1, 0].plot(range(mid, len(train_losses)), train_losses[mid:], label='Train Loss', linewidth=2, color='#1f77b4')
    axes[1, 0].plot(range(mid, len(val_losses)), val_losses[mid:], label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[1, 0].set_title('训练/验证 Loss (后半段)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(range(mid, len(train_accs)), train_accs[mid:], label='Train Acc', linewidth=2, color='#1f77b4')
    axes[1, 1].plot(range(mid, len(val_accs)), val_accs[mid:], label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[1, 1].set_title('训练/验证 准确率 (后半段)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_history_optimized.png', dpi=300, bbox_inches='tight')
    print("训练历史曲线已保存为 training_history_optimized.png")

# ========== 主函数 ==========
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    image_paths, labels, _ = load_data()
    
    # 数据划分（不变）
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.25, random_state=42, stratify=labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
    print(f"\n数据划分: 训练集 {len(train_paths)}, 验证集 {len(val_paths)}, 测试集 {len(test_paths)}")
    
    # ========== 关键改动3：恢复适度数据增强（轻量且有效） ==========
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),  # 轻量色彩抖动
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),              # 模拟轻微失焦
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = CervicalDataset(train_paths, train_labels, train_transform)
    val_dataset = CervicalDataset(val_paths, val_labels, val_test_transform)
    test_dataset = CervicalDataset(test_paths, test_labels, val_test_transform)
    
    # 采样器（保持不变，但移除*2过采样）
    train_labels_np = np.array(train_labels)
    class_sample_counts = np.array([np.sum(train_labels_np == i) for i in range(7)])
    weights = 1. / torch.tensor(class_sample_counts, dtype=torch.float)
    severe_idx = CLASS_NAMES['severe_dysplastic']
    weights[severe_idx] *= 1.5
    samples_weights = weights[train_labels_np]
    sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)
    
    # DataLoader（batch_size=64, num_workers=4）
    train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)
    
    # ========== 关键改动4：使用预训练ResNet50模型 ==========
    model = CervicalCNN_Pretrained(num_classes=7, pretrained_path='resnet50.pth')
    print("\n模型结构（已替换为ResNet50，冻结前3个stage）:")
    print(model)
    
    # 损失函数（使用修正后的CLASS_WEIGHTS）
    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(device), gamma=2.0, reduction='mean')
    
    # 优化器
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    # ========== 关键改动5：使用OneCycleLR调度器 ==========
    total_steps = len(train_loader) * 60  # 60 epochs
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=0.001, total_steps=total_steps, pct_start=0.3
    )
    
    # 训练（60 epochs）
    model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler, 
        num_epochs=60, device=device
    )
    
    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    evaluate_model(model, test_loader, device=device)
    
    print("\n" + "=" * 100)
    print("【优化完成】迁移学习(ResNet50) + 修正权重 + CutMix + OneCycleLR + 适度增强")
    print("=" * 100)

if __name__ == "__main__":
    main()

使用设备: cuda
正在加载数据集...
总共加载 1834 张图像

各类别样本分布:
  浅表鳞状上皮: 148
  中度鳞状上皮: 140
  柱状上皮: 196
  轻度发育不良: 364
  中度发育不良: 292
  重度发育不良: 394
  原位癌: 300

数据划分: 训练集 1375, 验证集 229, 测试集 230
成功加载本地预训练权重: resnet50.pth

模型结构（已替换为ResNet50，冻结前3个stage）:
CervicalCNN_Pretrained(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        

In [3]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import RandAugment
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0,
    'normal_intermediate': 1,
    'normal_columnar': 2,
    'light_dysplastic': 3,
    'moderate_dysplastic': 4,
    'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}

CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}

CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮',
    'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮',
    'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良',
    'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 根据当前性能重新调整类别权重（强化中/重度发育不良和原位癌）
CLASS_WEIGHTS = torch.tensor([1.2, 1.2, 1.8, 1.8, 3.0, 4.0, 2.5], dtype=torch.float32)

# ========== 标签平滑 Focal Loss ==========
class FocalLossWithLabelSmoothing(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs, dim=-1)
        n_classes = inputs.size(-1)
        # 平滑标签
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        ce_loss = -torch.sum(true_dist * log_probs, dim=-1)
        # 计算 pt
        pt = torch.exp(-ce_loss)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_weight = alpha_t * focal_weight
        loss = focal_weight * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# ========== 数据集类（不变） ==========
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

def load_data():
    print("正在加载数据集...")
    image_paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总共加载 {len(image_paths)} 张图像")
    class_counts = {}
    for label in labels:
        class_name = CLASS_NAMES_REV[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    print("\n各类别样本分布:")
    for class_name, count in class_counts.items():
        print(f"  {CLASS_NAMES_CN[class_name]}: {count}")
    return image_paths, labels, class_counts

# ========== 混合增强函数（组合MixUp和CutMix） ==========
def mixup_data(x, y, alpha=0.5):
    """MixUp增强"""
    if np.random.rand() > 0.5:
        return x, y, y, 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def cutmix_data(x, y, alpha=0.5):
    """CutMix增强（原函数）"""
    if np.random.rand() > 0.5:
        return x, y, y, 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def get_mixed_data(x, y):
    """随机选择MixUp或CutMix"""
    if np.random.rand() < 0.5:
        return mixup_data(x, y, alpha=0.5)
    else:
        return cutmix_data(x, y, alpha=0.5)

# ========== 使用EfficientNet-B4作为骨干 ==========
class CervicalModel(nn.Module):
    def __init__(self, num_classes=7, model_name='efficientnet_b4'):
        super().__init__()
        # 加载预训练EfficientNet-B4（会自动下载权重）
        self.backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        # 替换分类头
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        # 解冻所有层（用较小学习率微调）
        for param in self.backbone.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)

# 如果需要使用ResNet101（备用），取消注释以下类
# class CervicalModel(nn.Module):
#     def __init__(self, num_classes=7):
#         super().__init__()
#         self.backbone = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
#         in_features = self.backbone.fc.in_features
#         self.backbone.fc = nn.Sequential(
#             nn.Dropout(0.3),
#             nn.Linear(in_features, 512),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(512, num_classes)
#         )
#         for param in self.backbone.parameters():
#             param.requires_grad = True
#     def forward(self, x): return self.backbone(x)

# ========== 训练函数（支持混合增强 + 余弦退火重启） ==========
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, device='cuda', patience=20):
    model.to(device)
    best_val_f1 = 0.0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    wait = 0  # 早停计数器

    print(f"\n开始训练，共 {num_epochs} 轮...")
    print("使用标签平滑Focal Loss + 混合增强(MixUp/CutMix) + 余弦退火重启 + 解冻所有层")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            # 随机选择MixUp或CutMix
            inputs, labels_a, labels_b, lam = get_mixed_data(inputs, labels)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_train / total_train
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)

        # 验证
        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = correct_val / total_val
        _, _, f1, _ = precision_recall_fscore_support(all_val_labels, all_val_preds, average='weighted', zero_division=0)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {f1:.4f}")

        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = val_acc
            best_epoch = epoch
            wait = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': f1
            }, 'best_cervical_model_ultimate.pth')
            print(f"  → 保存最佳模型 (Epoch {best_epoch+1}, Val F1: {best_val_f1:.4f}, Acc: {best_val_acc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停触发，停止训练 at epoch {epoch+1}")
                break

        # 调度器更新（CosineAnnealingWarmRestarts在epoch后step）
        scheduler.step()

    print(f"\n训练完成！最佳 Epoch: {best_epoch+1}, Val F1: {best_val_f1:.4f}, Val Acc: {best_val_acc:.4f}")
    return model, train_losses, val_losses, train_accs, val_accs

# ========== 评估函数（适配新模型） ==========
def evaluate_model(model, test_loader, device='cuda'):
    checkpoint = torch.load('best_cervical_model_ultimate.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    print("\n开始评估模型...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    print("\n" + "=" * 100)
    print("【七分类详细评估指标】")
    print("=" * 100)
    report = classification_report(all_labels, all_preds, 
                               target_names=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                               digits=4, zero_division=0)
    print(report)

    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, zero_division=0)
    print("\n【各类别详细指标】")
    print(f"{'类别':<20} {'精确率(Precision)':<15} {'召回率(Recall)':<15} {'F1分数':<10} {'样本数':<10}")
    print("-" * 100)
    for i, class_name in enumerate(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES_CN[class_name]:<20} {precision[i]:<15.4f} {recall[i]:<15.4f} {f1[i]:<10.4f} {support[i]:<10}")

    binary_preds = [1 if p >= 3 else 0 for p in all_preds]
    binary_labels = [1 if l >= 3 else 0 for l in all_labels]
    print("\n" + "=" * 100)
    print("【二分类（正常/异常）评估指标】")
    print("=" * 100)
    binary_report = classification_report(binary_labels, binary_preds, target_names=['正常', '异常'], digits=4, zero_division=0)
    print(binary_report)

    plt.figure(figsize=(14, 12))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()], 
                yticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                annot_kws={'size': 12})
    plt.title('混淆矩阵 - 七分类 (终极优化)', fontsize=16, fontweight='bold')
    plt.xlabel('预测类别', fontsize=14)
    plt.ylabel('真实类别', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix_ultimate.png', dpi=300, bbox_inches='tight')
    print("\n混淆矩阵已保存为 confusion_matrix_ultimate.png")
    return all_preds, all_labels, all_probs

# ========== 绘图函数（不变） ==========
def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2, color='#1f77b4')
    axes[0, 0].plot(val_losses, label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[0, 0].set_title('训练/验证 Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(train_accs, label='Train Acc', linewidth=2, color='#1f77b4')
    axes[0, 1].plot(val_accs, label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[0, 1].set_title('训练/验证 准确率', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
    mid = max(0, len(train_losses) // 2)
    if mid > 0:
        axes[1, 0].plot(range(mid, len(train_losses)), train_losses[mid:], label='Train Loss', linewidth=2, color='#1f77b4')
        axes[1, 0].plot(range(mid, len(val_losses)), val_losses[mid:], label='Val Loss', linewidth=2, color='#ff7f0e')
        axes[1, 0].set_title('训练/验证 Loss (后半段)', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
        axes[1, 1].plot(range(mid, len(train_accs)), train_accs[mid:], label='Train Acc', linewidth=2, color='#1f77b4')
        axes[1, 1].plot(range(mid, len(val_accs)), val_accs[mid:], label='Val Acc', linewidth=2, color='#ff7f0e')
        axes[1, 1].set_title('训练/验证 准确率 (后半段)', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_history_ultimate.png', dpi=300, bbox_inches='tight')
    print("训练历史曲线已保存为 training_history_ultimate.png")

# ========== 主函数 ==========
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    image_paths, labels, _ = load_data()

    # 数据划分（保持原比例）
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.25, random_state=42, stratify=labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
    print(f"\n数据划分: 训练集 {len(train_paths)}, 验证集 {len(val_paths)}, 测试集 {len(test_paths)}")

    # ========== 终极数据增强：使用 RandAugment + 基础变换 ==========
    train_transform = transforms.Compose([
        transforms.Resize((420, 420)),          # 放大后裁剪至384
        transforms.RandomCrop((384, 384)),
        RandAugment(num_ops=2, magnitude=9),   # 自动增强
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((420, 420)),
        transforms.CenterCrop((384, 384)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = CervicalDataset(train_paths, train_labels, train_transform)
    val_dataset = CervicalDataset(val_paths, val_labels, val_test_transform)
    test_dataset = CervicalDataset(test_paths, test_labels, val_test_transform)

    # 采样器（保留，但可考虑不使用，因为使用了混合增强）
    train_labels_np = np.array(train_labels)
    class_sample_counts = np.array([np.sum(train_labels_np == i) for i in range(7)])
    weights = 1. / torch.tensor(class_sample_counts, dtype=torch.float)
    severe_idx = CLASS_NAMES['severe_dysplastic']
    weights[severe_idx] *= 1.8  # 额外强化
    samples_weights = weights[train_labels_np]
    sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

    # 注意：batch_size 根据显存调整，384分辨率下建议32或24
    batch_size = 24 if torch.cuda.is_available() else 16
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 模型
    model = CervicalModel(num_classes=7)  # 使用EfficientNet-B4
    print("\n模型结构（EfficientNet-B4，全部层可训练）:")
    print(model)

    # 损失函数（标签平滑Focal Loss）
    criterion = FocalLossWithLabelSmoothing(alpha=CLASS_WEIGHTS.to(device), gamma=2.0, smoothing=0.1, reduction='mean')

    # 优化器：所有层使用较小学习率
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=5e-5)

    # 调度器：余弦退火重启（T_0=10, T_mult=2）
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    # 训练（100轮，早停patience=25）
    model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        num_epochs=100, device=device, patience=25
    )

    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    evaluate_model(model, test_loader, device=device)

    print("\n" + "=" * 100)
    print("【终极优化完成】EfficientNet-B4 + 标签平滑Focal Loss + RandAugment + 混合增强 + 余弦退火重启")
    print("=" * 100)

if __name__ == "__main__":
    main()

使用设备: cuda
正在加载数据集...
总共加载 1834 张图像

各类别样本分布:
  浅表鳞状上皮: 148
  中度鳞状上皮: 140
  柱状上皮: 196
  轻度发育不良: 364
  中度发育不良: 292
  重度发育不良: 394
  原位癌: 300

数据划分: 训练集 1375, 验证集 229, 测试集 230


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:51<00:00, 1.53MB/s]



模型结构（EfficientNet-B4，全部层可训练）:
CervicalModel(
  (backbone): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
              (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(

OutOfMemoryError: CUDA out of memory. Tried to allocate 162.00 MiB. GPU 0 has a total capacty of 5.81 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 5.91 GiB memory in use. Of the allocated memory 5.30 GiB is allocated by PyTorch, and 280.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import RandAugment
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0,
    'normal_intermediate': 1,
    'normal_columnar': 2,
    'light_dysplastic': 3,
    'moderate_dysplastic': 4,
    'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}

CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}

CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮',
    'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮',
    'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良',
    'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 类别权重（基于之前最优表现微调）
CLASS_WEIGHTS = torch.tensor([1.2, 1.2, 1.8, 1.8, 3.0, 4.0, 2.5], dtype=torch.float32)

# ========== 标签平滑 Focal Loss ==========
class FocalLossWithLabelSmoothing(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs, dim=-1)
        n_classes = inputs.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        ce_loss = -torch.sum(true_dist * log_probs, dim=-1)
        pt = torch.exp(-ce_loss)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_weight = alpha_t * focal_weight
        loss = focal_weight * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# ========== 数据集类 ==========
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

def load_data():
    print("正在加载数据集...")
    image_paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总共加载 {len(image_paths)} 张图像")
    class_counts = {}
    for label in labels:
        class_name = CLASS_NAMES_REV[label]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    print("\n各类别样本分布:")
    for class_name, count in class_counts.items():
        print(f"  {CLASS_NAMES_CN[class_name]}: {count}")
    return image_paths, labels, class_counts

# ========== 混合增强 ==========
def mixup_data(x, y, alpha=0.5):
    if np.random.rand() > 0.5:
        return x, y, y, 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def cutmix_data(x, y, alpha=0.5):
    if np.random.rand() > 0.5:
        return x, y, y, 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def get_mixed_data(x, y):
    if np.random.rand() < 0.5:
        return mixup_data(x, y, alpha=0.5)
    else:
        return cutmix_data(x, y, alpha=0.5)

# ========== 使用EfficientNet-B4（修复显存版本） ==========
class CervicalModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        # 加载预训练EfficientNet-B4
        self.backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        # 全部层参与微调（使用较小学习率）
        for param in self.backbone.parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)

# ========== 训练函数（支持混合精度） ==========
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, device='cuda', patience=20):
    model.to(device)
    scaler = GradScaler()  # 混合精度缩放器
    
    best_val_f1 = 0.0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    wait = 0

    print(f"\n开始训练，共 {num_epochs} 轮...")
    print("使用标签平滑Focal Loss + 混合增强 + 余弦退火重启 + 混合精度训练")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs, labels_a, labels_b, lam = get_mixed_data(inputs, labels)

            optimizer.zero_grad()
            with autocast():  # 混合精度前向
                outputs = model(inputs)
                loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct_train / total_train
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc)

        # 验证
        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = correct_val / total_val
        _, _, f1, _ = precision_recall_fscore_support(all_val_labels, all_val_preds, average='weighted', zero_division=0)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {f1:.4f}")

        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = val_acc
            best_epoch = epoch
            wait = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': f1
            }, 'best_cervical_model_ultimate.pth')
            print(f"  → 保存最佳模型 (Epoch {best_epoch+1}, Val F1: {best_val_f1:.4f}, Acc: {best_val_acc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停触发，停止训练 at epoch {epoch+1}")
                break

        scheduler.step()

    print(f"\n训练完成！最佳 Epoch: {best_epoch+1}, Val F1: {best_val_f1:.4f}, Val Acc: {best_val_acc:.4f}")
    return model, train_losses, val_losses, train_accs, val_accs

# ========== 评估函数 ==========
def evaluate_model(model, test_loader, device='cuda'):
    checkpoint = torch.load('best_cervical_model_ultimate.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    print("\n开始评估模型...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            with autocast():
                outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    print("\n" + "=" * 100)
    print("【七分类详细评估指标】")
    print("=" * 100)
    report = classification_report(all_labels, all_preds, 
                               target_names=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                               digits=4, zero_division=0)
    print(report)

    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, zero_division=0)
    print("\n【各类别详细指标】")
    print(f"{'类别':<20} {'精确率(Precision)':<15} {'召回率(Recall)':<15} {'F1分数':<10} {'样本数':<10}")
    print("-" * 100)
    for i, class_name in enumerate(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES_CN[class_name]:<20} {precision[i]:<15.4f} {recall[i]:<15.4f} {f1[i]:<10.4f} {support[i]:<10}")

    binary_preds = [1 if p >= 3 else 0 for p in all_preds]
    binary_labels = [1 if l >= 3 else 0 for l in all_labels]
    print("\n" + "=" * 100)
    print("【二分类（正常/异常）评估指标】")
    print("=" * 100)
    binary_report = classification_report(binary_labels, binary_preds, target_names=['正常', '异常'], digits=4, zero_division=0)
    print(binary_report)

    plt.figure(figsize=(14, 12))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()], 
                yticklabels=[CLASS_NAMES_CN[name] for name in CLASS_NAMES.keys()],
                annot_kws={'size': 12})
    plt.title('混淆矩阵 - 七分类 (终极优化-显存修复版)', fontsize=16, fontweight='bold')
    plt.xlabel('预测类别', fontsize=14)
    plt.ylabel('真实类别', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix_ultimate.png', dpi=300, bbox_inches='tight')
    print("\n混淆矩阵已保存为 confusion_matrix_ultimate.png")
    return all_preds, all_labels, all_probs

# ========== 绘图函数 ==========
def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2, color='#1f77b4')
    axes[0, 0].plot(val_losses, label='Val Loss', linewidth=2, color='#ff7f0e')
    axes[0, 0].set_title('训练/验证 Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(train_accs, label='Train Acc', linewidth=2, color='#1f77b4')
    axes[0, 1].plot(val_accs, label='Val Acc', linewidth=2, color='#ff7f0e')
    axes[0, 1].set_title('训练/验证 准确率', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)
    mid = max(0, len(train_losses) // 2)
    if mid > 0:
        axes[1, 0].plot(range(mid, len(train_losses)), train_losses[mid:], label='Train Loss', linewidth=2, color='#1f77b4')
        axes[1, 0].plot(range(mid, len(val_losses)), val_losses[mid:], label='Val Loss', linewidth=2, color='#ff7f0e')
        axes[1, 0].set_title('训练/验证 Loss (后半段)', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)
        axes[1, 1].plot(range(mid, len(train_accs)), train_accs[mid:], label='Train Acc', linewidth=2, color='#1f77b4')
        axes[1, 1].plot(range(mid, len(val_accs)), val_accs[mid:], label='Val Acc', linewidth=2, color='#ff7f0e')
        axes[1, 1].set_title('训练/验证 准确率 (后半段)', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_history_ultimate.png', dpi=300, bbox_inches='tight')
    print("训练历史曲线已保存为 training_history_ultimate.png")

# ========== 主函数 ==========
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    # 检查显存，自动调整配置
    if torch.cuda.is_available():
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # GB
        print(f"GPU总显存: {total_memory:.2f} GB")
        if total_memory < 6:
            BATCH_SIZE = 12
            IMG_SIZE = 256
            print("显存 < 6GB，使用 batch_size=12, 分辨率=256")
        elif total_memory < 10:
            BATCH_SIZE = 16
            IMG_SIZE = 288
            print("显存 6~10GB，使用 batch_size=16, 分辨率=288")
        else:
            BATCH_SIZE = 24
            IMG_SIZE = 384
            print("显存 >= 10GB，使用 batch_size=24, 分辨率=384")
    else:
        BATCH_SIZE = 8
        IMG_SIZE = 224
        print("CPU模式，使用 batch_size=8, 分辨率=224")

    image_paths, labels, _ = load_data()

    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.25, random_state=42, stratify=labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
    print(f"\n数据划分: 训练集 {len(train_paths)}, 验证集 {len(val_paths)}, 测试集 {len(test_paths)}")

    # 数据增强（分辨率自适应）
    resize_size = int(IMG_SIZE * 1.25)  # 放大后裁剪
    train_transform = transforms.Compose([
        transforms.Resize((resize_size, resize_size)),
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        RandAugment(num_ops=2, magnitude=9),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((resize_size, resize_size)),
        transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = CervicalDataset(train_paths, train_labels, train_transform)
    val_dataset = CervicalDataset(val_paths, val_labels, val_test_transform)
    test_dataset = CervicalDataset(test_paths, test_labels, val_test_transform)

    # 采样器（强化困难类别）
    train_labels_np = np.array(train_labels)
    class_sample_counts = np.array([np.sum(train_labels_np == i) for i in range(7)])
    weights = 1. / torch.tensor(class_sample_counts, dtype=torch.float)
    severe_idx = CLASS_NAMES['severe_dysplastic']
    weights[severe_idx] *= 1.8
    samples_weights = weights[train_labels_np]
    sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    # 模型
    model = CervicalModel(num_classes=7)
    print(f"\n模型结构（EfficientNet-B4，输入尺寸 {IMG_SIZE}x{IMG_SIZE}，全部层可训练）:")
    print(model)

    # 损失函数
    criterion = FocalLossWithLabelSmoothing(alpha=CLASS_WEIGHTS.to(device), gamma=2.0, smoothing=0.1, reduction='mean')

    # 优化器（学习率根据batch size调整）
    lr = 3e-5 if BATCH_SIZE <= 16 else 5e-5
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5)

    # 调度器
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    # 训练
    model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        num_epochs=100, device=device, patience=25
    )

    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    evaluate_model(model, test_loader, device=device)

    print("\n" + "=" * 100)
    print("【终极优化 + 显存修复完成】EfficientNet-B4 + 自适应分辨率/BS + 混合精度 + RandAugment + 混合增强")
    print("=" * 100)

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用设备: cuda
GPU总显存: 6.24 GB
显存 6~10GB，使用 batch_size=16, 分辨率=288
正在加载数据集...
总共加载 1834 张图像

各类别样本分布:
  浅表鳞状上皮: 148
  中度鳞状上皮: 140
  柱状上皮: 196
  轻度发育不良: 364
  中度发育不良: 292
  重度发育不良: 394
  原位癌: 300

数据划分: 训练集 1375, 验证集 229, 测试集 230


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:14<00:00, 5.31MB/s]



模型结构（EfficientNet-B4，输入尺寸 288x288，全部层可训练）:
CervicalModel(
  (backbone): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
              (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
              (activ

In [2]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0, 'normal_intermediate': 1, 'normal_columnar': 2,
    'light_dysplastic': 3, 'moderate_dysplastic': 4, 'severe_dysplastic': 5, 'carcinoma_in_situ': 6
}
CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮', 'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮', 'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良', 'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 类别权重（强化中/重度 + 原位癌）
CLASS_WEIGHTS = torch.tensor([1.0, 1.0, 1.5, 1.5, 2.8, 3.5, 2.2], dtype=torch.float32)

class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("正在加载 Herlev 数据集...")
    image_paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总计 {len(image_paths)} 张图片")
    return image_paths, labels

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            ce_loss = alpha_t * ce_loss
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# 定义模型：仅微调最后两层 + 全连接层
class CervicalResNet50(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # 冻结所有层
        for param in self.backbone.parameters():
            param.requires_grad = False
        # 解冻 layer4 和 fc（适应细粒度特征）
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        # 新层使用较大的学习率，需要单独分组
        self.fc_params = list(self.backbone.fc.parameters())
        self.layer4_params = list(self.backbone.layer4.parameters())
    def forward(self, x):
        return self.backbone(x)

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=150):
    model.to(device)
    best_f1, best_acc, best_epoch = 0.0, 0.0, 0
    wait = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    
    print("\n开始稳定训练（仅微调 Layer4 + FC）...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()
        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)
        train_losses.append(train_loss); train_accs.append(train_acc)
        
        # 验证
        model.eval()
        val_loss, correct_val, total_val = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbs)
                val_loss += loss.item() * imgs.size(0)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())
        val_acc = correct_val / total_val
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss); val_accs.append(val_acc)
        _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1, best_acc, best_epoch = f1, val_acc, epoch
            wait = 0
            torch.save(model.state_dict(), 'best_herlev_resnet50.pth')
            print(f"  >>> 保存最佳 (F1: {f1:.4f})")
        else:
            wait += 1
            if wait >= 30:
                print(f"早停于 {epoch+1} 轮")
                break
        scheduler.step(val_loss)  # ReduceLROnPlateau
    return model, train_losses, val_losses, train_accs, val_accs

def evaluate(model, test_loader, device):
    model.load_state_dict(torch.load('best_herlev_resnet50.pth'))
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, pred = torch.max(outputs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbs.numpy())
    print("\n" + "="*80)
    print("【七分类最终评估】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using {device}")
    
    paths, labels = load_data()
    X_train, X_temp, y_train, y_temp = train_test_split(paths, labels, test_size=0.25, stratify=labels, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
    print(f"训练: {len(X_train)}, 验证: {len(X_val)}, 测试: {len(X_test)}")
    
    # 温和增强（专为细胞图像设计）
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.15, 0.15, 0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    test_tf = transforms.Compose([
        transforms.Resize((256,256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    
    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds = CervicalDataset(X_val, y_val, test_tf)
    test_ds = CervicalDataset(X_test, y_test, test_tf)
    
    # Weighted Sampler
    y_np = np.array(y_train)
    class_counts = np.array([np.sum(y_np==i) for i in range(7)])
    weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    weights[5] *= 2.0  # 重度发育不良
    sampler = WeightedRandomSampler(weights[y_np], len(y_np), replacement=True)
    
    BATCH = 32
    train_loader = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_ds, BATCH, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_ds, BATCH, shuffle=False, num_workers=4)
    
    model = CervicalResNet50(7)
    
    # 分组学习率：新层（FC）用 1e-3，Layer4 用 1e-4
    optimizer = optim.AdamW([
        {'params': model.layer4_params, 'lr': 1e-4},
        {'params': model.fc_params, 'lr': 1e-3}
    ], weight_decay=1e-4)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=12, verbose=True)
    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(device), gamma=2.0)
    
    model, _, _, _, _ = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=150)
    evaluate(model, test_loader, device)

if __name__ == "__main__":
    main()

Using cuda
正在加载 Herlev 数据集...
总计 1834 张图片
训练: 1375, 验证: 229, 测试: 230


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [01:18<00:00, 1.30MB/s]



开始稳定训练（仅微调 Layer4 + FC）...
Epoch 1/150 | Train Acc: 0.3280 Loss: 3.0244 | Val Acc: 0.4148 F1: 0.3289
  >>> 保存最佳 (F1: 0.3289)
Epoch 2/150 | Train Acc: 0.4575 Loss: 2.1254 | Val Acc: 0.4934 F1: 0.4854
  >>> 保存最佳 (F1: 0.4854)
Epoch 3/150 | Train Acc: 0.5542 Loss: 1.7138 | Val Acc: 0.4978 F1: 0.4910
  >>> 保存最佳 (F1: 0.4910)
Epoch 4/150 | Train Acc: 0.6007 Loss: 1.6288 | Val Acc: 0.5328 F1: 0.5281
  >>> 保存最佳 (F1: 0.5281)
Epoch 5/150 | Train Acc: 0.6095 Loss: 1.5394 | Val Acc: 0.5502 F1: 0.5336
  >>> 保存最佳 (F1: 0.5336)
Epoch 6/150 | Train Acc: 0.6080 Loss: 1.5466 | Val Acc: 0.5677 F1: 0.5498
  >>> 保存最佳 (F1: 0.5498)
Epoch 7/150 | Train Acc: 0.6349 Loss: 1.4620 | Val Acc: 0.5983 F1: 0.6002
  >>> 保存最佳 (F1: 0.6002)
Epoch 8/150 | Train Acc: 0.6509 Loss: 1.4492 | Val Acc: 0.5633 F1: 0.5685
Epoch 9/150 | Train Acc: 0.6545 Loss: 1.4639 | Val Acc: 0.5590 F1: 0.5622
Epoch 10/150 | Train Acc: 0.6727 Loss: 1.3900 | Val Acc: 0.5721 F1: 0.5732
Epoch 11/150 | Train Acc: 0.6691 Loss: 1.2900 | Val Acc: 0.5677

In [3]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 路径和类别 ----------
DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0,
    'normal_intermediate': 1,
    'normal_columnar': 2,
    'light_dysplastic': 3,
    'moderate_dysplastic': 4,
    'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}
CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮',
    'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮',
    'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良',
    'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 类别权重（强化困难类别）
CLASS_WEIGHTS = torch.tensor([1.0, 1.0, 1.5, 1.5, 2.8, 3.8, 2.2], dtype=torch.float32)

# ---------- 数据集 ----------
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("正在加载 Herlev 数据集...")
    image_paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    image_paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总计 {len(image_paths)} 张图片")
    return image_paths, labels

# ---------- 损失函数：Focal Loss（无标签平滑，避免复杂） ----------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            ce_loss = alpha_t * ce_loss
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# ---------- 模型：冻结全部骨干，只训练 FC ----------
class CervicalResNet50_FCOnly(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        # 加载预训练 ResNet50
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # 冻结所有层
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 替换分类头（新层自动 requires_grad=True）
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.6),          # 更强的 dropout
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        # 保存 fc 参数以便优化器使用
        self.fc_params = list(self.backbone.fc.parameters())

    def forward(self, x):
        return self.backbone(x)

# ---------- 训练函数 ----------
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=100, patience=20):
    model.to(device)
    best_f1, best_acc, best_epoch = 0.0, 0.0, 0
    wait = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    print("\n开始训练（冻结骨干，仅微调 FC 层）...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbs)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()

        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # 验证
        model.eval()
        val_loss, correct_val, total_val = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbs)
                val_loss += loss.item() * imgs.size(0)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())

        val_acc = correct_val / total_val
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1, best_acc, best_epoch = f1, val_acc, epoch
            wait = 0
            torch.save(model.state_dict(), 'best_herlev_fc_only.pth')
            print(f"  >>> 保存最佳 (F1: {f1:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停于 {epoch+1} 轮")
                break

        scheduler.step(val_loss)  # ReduceLROnPlateau

    print(f"训练完成！最佳验证 F1: {best_f1:.4f} (Epoch {best_epoch+1})")
    return model, train_losses, val_losses, train_accs, val_accs

# ---------- 评估函数 ----------
def evaluate(model, test_loader, device):
    model.load_state_dict(torch.load('best_herlev_fc_only.pth'))
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, pred = torch.max(outputs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbs.numpy())

    print("\n" + "=" * 80)
    print("【七分类最终评估（仅微调 FC）】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))

    # 混淆矩阵
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(CLASS_NAMES_CN.values()),
                yticklabels=list(CLASS_NAMES_CN.values()))
    plt.title('Confusion Matrix (FC only)')
    plt.tight_layout()
    plt.savefig('confusion_matrix_fc_only.png')
    print("混淆矩阵已保存为 confusion_matrix_fc_only.png")

# ---------- 主函数 ----------
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 加载数据
    paths, labels = load_data()
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.25, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    print(f"训练: {len(X_train)}, 验证: {len(X_val)}, 测试: {len(X_test)}")

    # 数据增强（温和，避免破坏细胞结构）
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(15),          # 减小旋转角度
        transforms.ColorJitter(0.1, 0.1, 0.1),  # 轻微色彩抖动
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    test_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds   = CervicalDataset(X_val, y_val, test_tf)
    test_ds  = CervicalDataset(X_test, y_test, test_tf)

    # Weighted Random Sampler
    y_np = np.array(y_train)
    class_counts = np.array([np.sum(y_np == i) for i in range(7)])
    weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    weights[5] *= 2.0  # 重度发育不良加倍
    sample_weights = weights[y_np]
    sampler = WeightedRandomSampler(sample_weights, len(y_np), replacement=True)

    BATCH_SIZE = 32
    train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=4)

    # 模型
    model = CervicalResNet50_FCOnly(num_classes=7)
    print("\n模型结构：冻结全部骨干，仅训练 FC 层。")

    # 损失函数
    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(device), gamma=2.0)

    # 优化器：只优化 FC 参数，使用较高权重衰减
    optimizer = optim.AdamW(model.fc_params, lr=1e-3, weight_decay=1e-2)

    # 调度器：ReduceLROnPlateau，更保守的 patience
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15, verbose=True)

    # 训练
    model, _, _, _, _ = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        device, epochs=100, patience=20
    )

    # 评估
    evaluate(model, test_loader, device)

    print("\n优化完成！请查看混淆矩阵和分类报告。")

if __name__ == "__main__":
    main()

Using device: cuda
正在加载 Herlev 数据集...
总计 1834 张图片
训练: 1375, 验证: 229, 测试: 230

模型结构：冻结全部骨干，仅训练 FC 层。

开始训练（冻结骨干，仅微调 FC 层）...
Epoch 1/100 | Train Acc: 0.2640 Loss: 3.5116 | Val Acc: 0.2533 F1: 0.1381
  >>> 保存最佳 (F1: 0.1381)
Epoch 2/100 | Train Acc: 0.3280 Loss: 2.8960 | Val Acc: 0.3450 F1: 0.2468
  >>> 保存最佳 (F1: 0.2468)
Epoch 3/100 | Train Acc: 0.3898 Loss: 2.6669 | Val Acc: 0.3843 F1: 0.3009
  >>> 保存最佳 (F1: 0.3009)
Epoch 4/100 | Train Acc: 0.3942 Loss: 2.6021 | Val Acc: 0.4192 F1: 0.3158
  >>> 保存最佳 (F1: 0.3158)
Epoch 5/100 | Train Acc: 0.4015 Loss: 2.5501 | Val Acc: 0.4410 F1: 0.3718
  >>> 保存最佳 (F1: 0.3718)
Epoch 6/100 | Train Acc: 0.3971 Loss: 2.5127 | Val Acc: 0.4410 F1: 0.3275
Epoch 7/100 | Train Acc: 0.4051 Loss: 2.4464 | Val Acc: 0.4017 F1: 0.3262
Epoch 8/100 | Train Acc: 0.4153 Loss: 2.4416 | Val Acc: 0.4410 F1: 0.3346
Epoch 9/100 | Train Acc: 0.3993 Loss: 2.4562 | Val Acc: 0.4454 F1: 0.3735
  >>> 保存最佳 (F1: 0.3735)
Epoch 10/100 | Train Acc: 0.4240 Loss: 2.4549 | Val Acc: 0.4498 F1

In [4]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim.lr_scheduler import LambdaLR
import warnings
warnings.filterwarnings('ignore')

# ---------------------------- 固定设置 ----------------------------
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'
CLASS_NAMES = {
    'normal_superficiel': 0, 'normal_intermediate': 1, 'normal_columnar': 2,
    'light_dysplastic': 3, 'moderate_dysplastic': 4, 'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}
CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮', 'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮', 'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良', 'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}
# 类别权重（根据样本数量反比 + 强化困难类）
CLASS_WEIGHTS = torch.tensor([1.0, 1.0, 1.5, 1.5, 2.5, 3.5, 2.0], dtype=torch.float32)

# ---------------------------- 数据集 ----------------------------
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("加载 Herlev 数据集...")
    paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总图片数: {len(paths)}")
    return paths, labels

# ---------------------------- 损失函数（Focal Loss） ----------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            ce_loss = alpha_t * ce_loss
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# ---------------------------- 模型：差异化微调 ----------------------------
class CervicalResNet50_FineTune(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # 先冻结所有层
        for param in self.backbone.parameters():
            param.requires_grad = False
        # 解冻 layer3 和 layer4（高语义层）
        for name, param in self.backbone.named_parameters():
            if name.startswith('layer3') or name.startswith('layer4'):
                param.requires_grad = True

        # 替换分类头（更深的头）
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        # 保存不同组的参数用于分组学习率
        self.fc_params = list(self.backbone.fc.parameters())
        self.layer3_params = []
        self.layer4_params = []
        for name, param in self.backbone.named_parameters():
            if name.startswith('layer3'):
                self.layer3_params.append(param)
            elif name.startswith('layer4'):
                self.layer4_params.append(param)

    def forward(self, x):
        return self.backbone(x)

# ---------------------------- 预热 + 余弦退火调度器 ----------------------------
def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            # 线性预热
            return epoch / warmup_epochs
        else:
            # 余弦衰减
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress)) * (1 - min_lr) + min_lr
    return LambdaLR(optimizer, lr_lambda)

# ---------------------------- 训练函数 ----------------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=80, patience=15):
    model.to(device)
    best_f1, best_epoch = 0.0, 0
    wait = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    print("\n开始差异化微调（layer3+layer4 低学习率，FC 高学习率）...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbs)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()

        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # 验证
        model.eval()
        val_loss, correct_val, total_val = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbs)
                val_loss += loss.item() * imgs.size(0)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())

        val_acc = correct_val / total_val
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch
            wait = 0
            torch.save(model.state_dict(), 'best_hervel_finetune.pth')
            print(f"  >>> 更新最佳模型 (F1: {f1:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停于 epoch {epoch+1}")
                break

        scheduler.step()  # LambdaLR 按 epoch 更新

    print(f"训练完成！最佳验证 F1: {best_f1:.4f} (Epoch {best_epoch+1})")
    return model, train_losses, val_losses, train_accs, val_accs

# ---------------------------- 评估函数 ----------------------------
def evaluate(model, test_loader, device):
    model.load_state_dict(torch.load('best_hervel_finetune.pth'))
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, pred = torch.max(outputs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbs.numpy())

    print("\n" + "=" * 80)
    print("【七分类最终评估（差异化微调）】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))

    # 混淆矩阵
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(CLASS_NAMES_CN.values()),
                yticklabels=list(CLASS_NAMES_CN.values()))
    plt.title('Confusion Matrix (Fine-tuned)')
    plt.tight_layout()
    plt.savefig('confusion_matrix_finetune.png')
    print("混淆矩阵已保存为 confusion_matrix_finetune.png")

# ---------------------------- 主函数 ----------------------------
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    paths, labels = load_data()
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.25, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    print(f"训练集: {len(X_train)}, 验证集: {len(X_val)}, 测试集: {len(X_test)}")

    # 数据增强（温和）
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    test_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds   = CervicalDataset(X_val, y_val, test_tf)
    test_ds  = CervicalDataset(X_test, y_test, test_tf)

    # Weighted Sampler
    y_np = np.array(y_train)
    class_counts = np.array([np.sum(y_np == i) for i in range(7)])
    weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    weights[5] *= 2.0  # 重度发育不良
    sample_weights = weights[y_np]
    sampler = WeightedRandomSampler(sample_weights, len(y_np), replacement=True)

    BATCH = 32
    train_loader = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=4)

    # 模型
    model = CervicalResNet50_FineTune(num_classes=7)
    print("\n模型结构：解冻 layer3 & layer4，FC 头扩增为 3 层。")

    # 分组学习率：骨干层低 lr，FC 层高 lr
    backbone_params = model.layer3_params + model.layer4_params
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},
        {'params': model.fc_params, 'lr': 1e-3}
    ], weight_decay=1e-4)

    # 预热 + 余弦衰减调度器（总 epoch=80，预热 5 轮）
    scheduler = get_warmup_cosine_scheduler(optimizer, warmup_epochs=5, total_epochs=80, min_lr=1e-6)

    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(device), gamma=2.0)

    model, _, _, _, _ = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        device, epochs=80, patience=18
    )

    evaluate(model, test_loader, device)

    print("\n优化完成！")

if __name__ == "__main__":
    main()

使用设备: cuda
加载 Herlev 数据集...
总图片数: 1834
训练集: 1375, 验证集: 229, 测试集: 230

模型结构：解冻 layer3 & layer4，FC 头扩增为 3 层。

开始差异化微调（layer3+layer4 低学习率，FC 高学习率）...
Epoch 1/80 | Train Acc: 0.1578 Loss: 3.6574 | Val Acc: 0.1790 F1: 0.0902
  >>> 更新最佳模型 (F1: 0.0902)
Epoch 2/80 | Train Acc: 0.2575 Loss: 3.2396 | Val Acc: 0.2140 F1: 0.0754
Epoch 3/80 | Train Acc: 0.2909 Loss: 2.7383 | Val Acc: 0.3275 F1: 0.1988
  >>> 更新最佳模型 (F1: 0.1988)
Epoch 4/80 | Train Acc: 0.4095 Loss: 2.2798 | Val Acc: 0.4498 F1: 0.3807
  >>> 更新最佳模型 (F1: 0.3807)
Epoch 5/80 | Train Acc: 0.4240 Loss: 2.1707 | Val Acc: 0.4105 F1: 0.3729
Epoch 6/80 | Train Acc: 0.4764 Loss: 1.8401 | Val Acc: 0.4760 F1: 0.4547
  >>> 更新最佳模型 (F1: 0.4547)
Epoch 7/80 | Train Acc: 0.4902 Loss: 1.8767 | Val Acc: 0.4323 F1: 0.3766
Epoch 8/80 | Train Acc: 0.5091 Loss: 1.7282 | Val Acc: 0.4978 F1: 0.4489
Epoch 9/80 | Train Acc: 0.5651 Loss: 1.5024 | Val Acc: 0.5415 F1: 0.5132
  >>> 更新最佳模型 (F1: 0.5132)
Epoch 10/80 | Train Acc: 0.5775 Loss: 1.5452 | Val Acc: 0.5852 F1:

In [5]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# ---------------------------- 固定设置 ----------------------------
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'
CLASS_NAMES = {
    'normal_superficiel': 0, 'normal_intermediate': 1, 'normal_columnar': 2,
    'light_dysplastic': 3, 'moderate_dysplastic': 4, 'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}
CLASS_NAMES_REV = {v: k for k, v in CLASS_NAMES.items()}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮', 'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮', 'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良', 'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}
CLASS_WEIGHTS = torch.tensor([1.0, 1.0, 1.5, 1.5, 2.5, 3.5, 2.0], dtype=torch.float32)

# ---------------------------- 数据集 ----------------------------
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("加载 Herlev 数据集...")
    paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总图片数: {len(paths)}")
    return paths, labels

# ---------------------------- Focal Loss (标签平滑版本) ----------------------------
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        log_probs = nn.functional.log_softmax(inputs, dim=-1)
        n_classes = inputs.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        ce_loss = -torch.sum(true_dist * log_probs, dim=-1)
        pt = torch.exp(-ce_loss)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets)
            focal_weight = alpha_t * focal_weight
        loss = focal_weight * ce_loss
        return loss.mean()

# ---------------------------- MixUp 辅助 ----------------------------
def mixup_data(x, y, alpha=0.4):
    if np.random.rand() > 0.5:
        return x, y, y, 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    lam = np.random.beta(alpha, alpha)
    x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

# ---------------------------- 模型：EfficientNet-B3 ----------------------------
class CervicalEfficientNetB3(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        # 解冻所有层（后续通过分组学习率控制微调幅度）
        for param in self.backbone.parameters():
            param.requires_grad = True
        # 替换分类头（更宽的头）
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        # 收集参数分组
        self.feature_params = []
        self.fc_params = []
        for name, param in self.backbone.named_parameters():
            if 'classifier' in name:
                self.fc_params.append(param)
            else:
                self.feature_params.append(param)

    def forward(self, x):
        return self.backbone(x)

# ---------------------------- 训练函数（带MixUp） ----------------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=80, patience=15):
    model.to(device)
    best_f1, best_epoch = 0.0, 0
    wait = 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    print("\n开始训练 EfficientNet-B3（全层微调 + MixUp）...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            # MixUp
            imgs, lbs_a, lbs_b, lam = mixup_data(imgs, lbs, alpha=0.4)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = lam * criterion(outputs, lbs_a) + (1 - lam) * criterion(outputs, lbs_b)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()

        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # 验证
        model.eval()
        val_loss, correct_val, total_val = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbs)
                val_loss += loss.item() * imgs.size(0)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())

        val_acc = correct_val / total_val
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch
            wait = 0
            torch.save(model.state_dict(), 'best_herlev_efb3.pth')
            print(f"  >>> 保存最佳 (F1: {f1:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停于 epoch {epoch+1}")
                break

        scheduler.step()  # 余弦退火重启在 epoch 后更新

    print(f"训练完成！最佳验证 F1: {best_f1:.4f} (Epoch {best_epoch+1})")
    return model, train_losses, val_losses, train_accs, val_accs

# ---------------------------- 评估函数 ----------------------------
def evaluate(model, test_loader, device):
    model.load_state_dict(torch.load('best_herlev_efb3.pth'))
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, pred = torch.max(outputs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbs.numpy())

    print("\n" + "=" * 80)
    print("【七分类最终评估 (EfficientNet-B3 + MixUp)】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(CLASS_NAMES_CN.values()),
                yticklabels=list(CLASS_NAMES_CN.values()))
    plt.title('Confusion Matrix (EfficientNet-B3)')
    plt.tight_layout()
    plt.savefig('confusion_matrix_efb3.png')
    print("混淆矩阵已保存为 confusion_matrix_efb3.png")

# ---------------------------- 主函数 ----------------------------
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    paths, labels = load_data()
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.25, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    print(f"训练: {len(X_train)}, 验证: {len(X_val)}, 测试: {len(X_test)}")

    # 数据增强（EfficientNet 需要 300x300 输入）
    train_tf = transforms.Compose([
        transforms.Resize((320, 320)),
        transforms.RandomCrop((300, 300)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    test_tf = transforms.Compose([
        transforms.Resize((320, 320)),
        transforms.CenterCrop(300),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds   = CervicalDataset(X_val, y_val, test_tf)
    test_ds  = CervicalDataset(X_test, y_test, test_tf)

    # Weighted Sampler
    y_np = np.array(y_train)
    class_counts = np.array([np.sum(y_np == i) for i in range(7)])
    weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    weights[5] *= 2.0
    sample_weights = weights[y_np]
    sampler = WeightedRandomSampler(sample_weights, len(y_np), replacement=True)

    BATCH = 24  # EfficientNet-B3 稍大，显存若不足可减至16
    train_loader = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=4)

    # 模型
    model = CervicalEfficientNetB3(num_classes=7)
    print("\n模型：EfficientNet-B3 全层微调，分类头 512→256→7。")

    # 分组学习率：特征层 1e-4，FC 层 1e-3
    optimizer = optim.AdamW([
        {'params': model.feature_params, 'lr': 1e-4},
        {'params': model.fc_params, 'lr': 1e-3}
    ], weight_decay=1e-4)

    # 余弦退火重启（T_0=10, T_mult=2）
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    criterion = FocalLossWithSmoothing(alpha=CLASS_WEIGHTS.to(device), gamma=2.0, smoothing=0.05)

    # 训练
    model, _, _, _, _ = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        device, epochs=80, patience=18
    )

    evaluate(model, test_loader, device)

    print("\n=== 如需进一步提升，请训练 3 个不同随机种子模型进行集成 ===")

if __name__ == "__main__":
    main()

使用设备: cuda
加载 Herlev 数据集...
总图片数: 1834
训练: 1375, 验证: 229, 测试: 230


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [10:36<00:00, 77.8kB/s]



模型：EfficientNet-B3 全层微调，分类头 512→256→7。

开始训练 EfficientNet-B3（全层微调 + MixUp）...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacty of 5.81 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 5.90 GiB memory in use. Of the allocated memory 5.18 GiB is allocated by PyTorch, and 194.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torch.optim.lr_scheduler import LambdaLR
import warnings
warnings.filterwarnings('ignore')

# --------------------------- 固定设置 ---------------------------
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0, 'normal_intermediate': 1, 'normal_columnar': 2,
    'light_dysplastic': 3, 'moderate_dysplastic': 4, 'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮', 'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮', 'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良', 'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# 类别权重（按样本数反比）
def compute_class_weights(labels):
    counts = np.bincount(labels)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * len(weights)  # 归一化
    return torch.tensor(weights, dtype=torch.float32)

# --------------------------- 数据集 ---------------------------
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("加载 Herlev 数据集...")
    paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总图片数: {len(paths)}")
    return paths, labels

# --------------------------- 模型（简单 ResNet50） ---------------------------
class SimpleResNet50(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # 冻结所有层
        for param in self.backbone.parameters():
            param.requires_grad = False
        # 只解冻 layer4 和 fc
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)
        # 保存参数分组
        self.backbone_params = list(self.backbone.layer4.parameters())
        self.fc_params = list(self.backbone.fc.parameters())
    def forward(self, x):
        return self.backbone(x)

# --------------------------- 训练函数 ---------------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=80):
    model.to(device)
    best_acc = 0.0
    best_epoch = 0
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()
        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)

        # 验证
        model.eval()
        correct_val, total_val = 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())
        val_acc = correct_val / total_val
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_stable_model.pth')
        scheduler.step()
    print(f"训练完成！最佳验证准确率: {best_acc:.4f} (Epoch {best_epoch+1})")
    return model

# --------------------------- 评估 ---------------------------
def evaluate(model, test_loader, device):
    model.load_state_dict(torch.load('best_stable_model.pth'))
    model.to(device).eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, pred = torch.max(outputs, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lbs.numpy())
    print("\n" + "="*80)
    print("【稳定版评估】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(CLASS_NAMES_CN.values()),
                yticklabels=list(CLASS_NAMES_CN.values()))
    plt.title('Confusion Matrix (Stable)')
    plt.tight_layout()
    plt.savefig('confusion_matrix_stable.png')
    print("混淆矩阵已保存")

# --------------------------- 主函数 ---------------------------
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    paths, labels = load_data()
    # 划分训练、验证、测试 (60%训练, 20%验证, 20%测试)
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.4, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    print(f"训练集: {len(X_train)}, 验证集: {len(X_val)}, 测试集: {len(X_test)}")

    # 数据增强（温和）
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.1, 0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    test_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds   = CervicalDataset(X_val, y_val, test_tf)
    test_ds  = CervicalDataset(X_test, y_test, test_tf)

    # 使用 WeightedRandomSampler 缓解不均衡
    y_np = np.array(y_train)
    class_weights = compute_class_weights(y_np)
    sample_weights = class_weights[y_np]
    sampler = WeightedRandomSampler(sample_weights, len(y_np), replacement=True)

    BATCH = 32
    train_loader = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=4)

    # 模型
    model = SimpleResNet50(num_classes=7)
    print("\n模型: ResNet50 (仅微调 layer4 + fc)")

    # 损失函数：标准交叉熵 + 类别权重
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    # 优化器：分组学习率
    optimizer = optim.AdamW([
        {'params': model.backbone_params, 'lr': 1e-5},   # 骨干极低学习率
        {'params': model.fc_params,       'lr': 1e-3}    # 新分类头高学习率
    ], weight_decay=1e-4)

    # 预热 + 余弦衰减调度器
    def warmup_cosine(epoch, warmup=5, total=80):
        if epoch < warmup:
            return (epoch + 1) / warmup
        else:
            progress = (epoch - warmup) / (total - warmup)
            return 0.5 * (1 + np.cos(np.pi * progress)) * 0.5 + 0.5  # 衰减到0.5倍初始

    scheduler = LambdaLR(optimizer, lr_lambda=lambda e: warmup_cosine(e, warmup=5, total=80))

    # 训练（80轮）
    model = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=80)

    # 评估
    evaluate(model, test_loader, device)

    print("\n稳定版训练完成！")

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用设备: cuda
加载 Herlev 数据集...
总图片数: 1834
训练集: 1100, 验证集: 367, 测试集: 367

模型: ResNet50 (仅微调 layer4 + fc)
Epoch 1/80 | Train Acc: 0.1845 Loss: 1.7896 | Val Acc: 0.1962
Epoch 2/80 | Train Acc: 0.3236 Loss: 1.4754 | Val Acc: 0.2670
Epoch 3/80 | Train Acc: 0.4355 Loss: 1.1710 | Val Acc: 0.2779
Epoch 4/80 | Train Acc: 0.5400 Loss: 0.9448 | Val Acc: 0.5123
Epoch 5/80 | Train Acc: 0.5818 Loss: 0.8673 | Val Acc: 0.5313
Epoch 6/80 | Train Acc: 0.5955 Loss: 0.8188 | Val Acc: 0.5286
Epoch 7/80 | Train Acc: 0.6200 Loss: 0.7590 | Val Acc: 0.4523
Epoch 8/80 | Train Acc: 0.6245 Loss: 0.7254 | Val Acc: 0.4823
Epoch 9/80 | Train Acc: 0.6609 Loss: 0.6401 | Val Acc: 0.5204
Epoch 10/80 | Train Acc: 0.6327 Loss: 0.7767 | Val Acc: 0.5068
Epoch 11/80 | Train Acc: 0.6755 Loss: 0.6340 | Val Acc: 0.5123
Epoch 12/80 | Train Acc: 0.6564 Loss: 0.6416 | Val Acc: 0.5695
Epoch 13/80 | Train Acc: 0.6518 Loss: 0.7005 | Val Acc: 0.4850
Epoch 14/80 | Train Acc: 0.7055 Loss: 0.5659 | Val Acc: 0.5368
Epoch 15/80 | Train Acc: 0

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import RandAugment
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

# ---------------------------- 固定设置 ----------------------------
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = True

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '实训任务3数据集'

CLASS_NAMES = {
    'normal_superficiel': 0, 'normal_intermediate': 1, 'normal_columnar': 2,
    'light_dysplastic': 3, 'moderate_dysplastic': 4, 'severe_dysplastic': 5,
    'carcinoma_in_situ': 6
}
CLASS_NAMES_CN = {
    'normal_superficiel': '浅表鳞状上皮', 'normal_intermediate': '中度鳞状上皮',
    'normal_columnar': '柱状上皮', 'light_dysplastic': '轻度发育不良',
    'moderate_dysplastic': '中度发育不良', 'severe_dysplastic': '重度发育不良',
    'carcinoma_in_situ': '原位癌'
}

# ---------------------------- 数据集 ----------------------------
class CervicalDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

def load_data():
    print("加载 Herlev 数据集...")
    paths, labels = [], []
    for class_name, class_idx in CLASS_NAMES.items():
        class_dir = os.path.join(DATA_DIR, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.bmp', '.jpg', '.png')):
                    paths.append(os.path.join(class_dir, f))
                    labels.append(class_idx)
    print(f"总图片数: {len(paths)}")
    return paths, labels

# ---------------------------- 模型（ResNet50 全部微调） ----------------------------
def create_model(num_classes=7):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4, inplace=False),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=False),
        nn.Dropout(0.3, inplace=False),
        nn.Linear(512, 256),
        nn.ReLU(inplace=False),
        nn.Dropout(0.2, inplace=False),
        nn.Linear(256, num_classes)
    )
    return model

# ---------------------------- 训练函数（混合精度） ----------------------------
def train_model(model, train_loader, val_loader, device, epochs=100):
    model.to(device)
    # 类别权重
    train_labels = train_loader.dataset.labels
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # 分组学习率
    backbone_params = []
    classifier_params = []
    for name, param in model.named_parameters():
        if 'fc' in name:
            classifier_params.append(param)
        else:
            backbone_params.append(param)

    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 1e-4},
        {'params': classifier_params, 'lr': 1e-3}
    ], weight_decay=1e-4)

    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    best_acc = 0.0
    best_epoch = 0
    patience = 15
    wait = 0

    # 混合精度（可选）
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    print("\n开始训练 ResNet50 (全部层微调, 输入224x224)...")
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbs in train_loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            optimizer.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, lbs)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(imgs)
                loss = criterion(outputs, lbs)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(outputs, 1)
            total += lbs.size(0)
            correct += (pred == lbs).sum().item()

        train_acc = correct / total
        train_loss = total_loss / len(train_loader.dataset)

        # 验证
        model.eval()
        correct_val, total_val = 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbs in val_loader:
                imgs, lbs = imgs.to(device), lbs.to(device)
                outputs = model(imgs)
                _, pred = torch.max(outputs, 1)
                total_val += lbs.size(0)
                correct_val += (pred == lbs).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(lbs.cpu().numpy())

        val_acc = correct_val / total_val
        _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} F1: {f1:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            wait = 0
            torch.save(model.state_dict(), 'best_resnet50_full.pth')
            print(f"  >>> 保存最佳模型 (Acc: {val_acc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"早停于 epoch {epoch+1}")
                break

        scheduler.step()

    print(f"训练完成！最佳验证准确率: {best_acc:.4f} (Epoch {best_epoch+1})")
    return model

# ---------------------------- TTA 预测 ----------------------------
def predict_with_tta(model, test_loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            imgs = imgs.to(device)
            # 原始
            probs_orig = torch.softmax(model(imgs), dim=1).cpu().numpy()
            # 水平翻转
            probs_hflip = torch.softmax(model(torch.flip(imgs, dims=[3])), dim=1).cpu().numpy()
            # 垂直翻转
            probs_vflip = torch.softmax(model(torch.flip(imgs, dims=[2])), dim=1).cpu().numpy()
            avg_probs = (probs_orig + probs_hflip + probs_vflip) / 3.0
            all_probs.append(avg_probs)
            all_labels.extend(lbs.numpy())
    return np.vstack(all_probs), np.array(all_labels)

def evaluate_tta(model, test_loader, device):
    model.load_state_dict(torch.load('best_resnet50_full.pth', map_location=device))
    model.to(device)
    probs, labels = predict_with_tta(model, test_loader, device)
    preds = np.argmax(probs, axis=1)

    print("\n" + "=" * 80)
    print("【最终评估 (ResNet50 全微调 + TTA)】")
    print(classification_report(labels, preds, target_names=list(CLASS_NAMES_CN.values()), digits=4))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(CLASS_NAMES_CN.values()),
                yticklabels=list(CLASS_NAMES_CN.values()))
    plt.title('ResNet50 Full Fine-tune + TTA')
    plt.tight_layout()
    plt.savefig('confusion_matrix_final_resnet50.png')
    print("混淆矩阵已保存为 confusion_matrix_final_resnet50.png")

    acc = np.mean(labels == preds)
    _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    print(f"\n最终准确率: {acc:.4f}, 加权F1: {f1:.4f}")
    return acc, f1

# ---------------------------- 主函数 ----------------------------
def main():
    # ========== 修改1: 清理显存 ==========
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"初始显存占用: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    paths, labels = load_data()
    X_train, X_temp, y_train, y_temp = train_test_split(
        paths, labels, test_size=0.4, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    print(f"训练集: {len(X_train)}, 验证集: {len(X_val)}, 测试集: {len(X_test)}")

    # 数据增强（使用 RandAugment，输入 224x224）
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((224, 224)),
        RandAugment(num_ops=2, magnitude=7),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    test_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_ds = CervicalDataset(X_train, y_train, train_tf)
    val_ds   = CervicalDataset(X_val, y_val, test_tf)
    test_ds  = CervicalDataset(X_test, y_test, test_tf)

    # WeightedRandomSampler
    y_np = np.array(y_train)
    class_counts = np.bincount(y_np)
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = weights[y_np]
    sampler = WeightedRandomSampler(sample_weights, len(y_np), replacement=True)

    # ========== 修改2: 减小 batch size 至 16 ==========
    BATCH = 16   # 原先为 32，以适应 6GB 显存
    train_loader = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=4)

    model = create_model(num_classes=7)
    print("\n模型: ResNet50 (全部层微调, 224x224)")

    model = train_model(model, train_loader, val_loader, device, epochs=100)

    acc, f1 = evaluate_tta(model, test_loader, device)
    print(f"\n🎯 最终结果: 准确率 = {acc:.4f}, F1 = {f1:.4f}")

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


初始显存占用: 0.00 GB
使用设备: cuda
加载 Herlev 数据集...
总图片数: 1834
训练集: 1100, 验证集: 367, 测试集: 367

模型: ResNet50 (全部层微调, 224x224)

开始训练 ResNet50 (全部层微调, 输入224x224)...
Epoch 1/100 | Train Acc: 0.3027 Loss: 1.5632 | Val Acc: 0.2807 F1: 0.2135
  >>> 保存最佳模型 (Acc: 0.2807)
Epoch 2/100 | Train Acc: 0.4509 Loss: 1.3282 | Val Acc: 0.4687 F1: 0.3782
  >>> 保存最佳模型 (Acc: 0.4687)
Epoch 3/100 | Train Acc: 0.5636 Loss: 1.1451 | Val Acc: 0.5232 F1: 0.4754
  >>> 保存最佳模型 (Acc: 0.5232)
Epoch 4/100 | Train Acc: 0.5609 Loss: 1.1486 | Val Acc: 0.4687 F1: 0.4251
Epoch 5/100 | Train Acc: 0.5927 Loss: 1.0993 | Val Acc: 0.5259 F1: 0.4747
  >>> 保存最佳模型 (Acc: 0.5259)
Epoch 6/100 | Train Acc: 0.6118 Loss: 1.0803 | Val Acc: 0.5395 F1: 0.5081
  >>> 保存最佳模型 (Acc: 0.5395)
Epoch 7/100 | Train Acc: 0.6482 Loss: 0.9939 | Val Acc: 0.5422 F1: 0.5042
  >>> 保存最佳模型 (Acc: 0.5422)
Epoch 8/100 | Train Acc: 0.6745 Loss: 0.9389 | Val Acc: 0.5777 F1: 0.5428
  >>> 保存最佳模型 (Acc: 0.5777)
Epoch 9/100 | Train Acc: 0.6964 Loss: 0.9143 | Val Acc: 0.5640 F1: